# CBraMod fine-tune on BBBD Exp 2 (Colab GPU)

Hours 2.5–3.5 of the Focus Detection Feasibility Gameplan. Run **only if Gate 2 passed**
(`results/cbramod_probe_exp2_our8.json` says `PASS`).

Runtime → Change runtime type → **GPU** (T4 or better).

**Before running**, upload the project folder to Google Drive as `MyDrive/EEG_hackathon/` with at least:
- `scripts/` (all .py files)
- `results/` (the Riemannian and probe JSONs, for the final table)
- `data/processed/exp2_our8/200hz/` and `data/processed/exp2_our8/cbramod_frozen_features.npz` (~400 MB)

If the data folder is missing, cell 3 rebuilds it from S3 (~1.8 GB download + ~5 min preprocessing).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/EEG_hackathon'
%cd $PROJECT
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install braindecode==2.0.0 huggingface_hub safetensors pyriemann mne remotezip

In [ ]:
import os
if not os.path.isdir('data/processed/exp2_our8/200hz'):
    !python scripts/download_bbbd.py --exp 2 --out /content/bbbd
    !python scripts/preprocess.py --bids /content/bbbd/experiment2
print(len(os.listdir('data/processed/exp2_our8/200hz')), 'participants')

Frozen probe (fast on GPU; reuses cached features if present). Prints Gate 2.

In [ ]:
!cd scripts && python cbramod_probe.py --data ../data/processed/exp2_our8 --out ../results

Fine-tune: one fold per split, early stopping, ≤10 epochs. Keeps fine-tuning only if ≥ 0.03 above the probe on the same fold.

In [ ]:
!cd scripts && python cbramod_finetune.py --data ../data/processed/exp2_our8 --out ../results --epochs 10

Results table

In [ ]:
!cd scripts && python results_table.py --results ../results